
# 06_advanced_tricky — Продвинутая подготовка к ML Engineer Interview

Этот ноутбук покрывает сложные темы Python, которые часто встречаются на собеседованиях уровня Middle+/Senior для ML Engineer:
- внутренности языка,
- производительность и память,
- архитектурные механизмы (дескрипторы, метаклассы),
- подводные камни поведения Python.

> Все объяснения даны на русском языке, а код — на Python.


## План

1. LRU cache (ручная реализация + `functools.lru_cache`)
2. Дескрипторы
3. Метаклассы
4. Модель памяти Python
5. Подсчёт ссылок
6. Сборщик мусора
7. Внутренности объектов
8. `__slots__`
9. Интернирование
10. Edge cases в Python
11. Big-O deep dive
12. Частые tricky-вопросы
13. Mock Interview (20 сложных вопросов)

## Тема 1: LRU Cache (ручная реализация + functools.lru_cache)

### 1) Глубокая теория
LRU (Least Recently Used) — стратегия вытеснения, при которой удаляется **наименее недавно использованный** элемент.

Почему важно для ML Engineer:
- кэширование фичей/предобработки;
- повторные вычисления дорогих функций (парсинг, токенизация, инференс мелких блоков);
- контроль latency в online сервисах.

Классическая LRU-структура сочетает:
1. `dict` для O(1) доступа по ключу;
2. двусвязный список (или `OrderedDict`) для O(1) обновления «свежести».

`functools.lru_cache` — готовая и оптимизированная реализация, но важно понимать ограничения:
- ключ должен быть хешируемым;
- функция должна быть детерминирована для корректного кэширования;
- не всегда потокобезопасна с точки зрения логики предметной области.

In [ ]:
from collections import OrderedDict
from functools import lru_cache
import time

class LRUCacheManual:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

    def __repr__(self):
        return f"LRUCacheManual({self.data})"

cache = LRUCacheManual(3)
for i in [1, 2, 3, 1, 4, 2, 5]:
    if cache.get(i) is None:
        cache.put(i, i * i)
print(cache)

@lru_cache(maxsize=128)
def expensive_fn(x: int) -> int:
    time.sleep(0.01)
    return x * x

start = time.perf_counter()
for _ in range(2):
    for i in [1, 2, 3, 2, 1, 4, 5, 4]:
        expensive_fn(i)
elapsed = time.perf_counter() - start
print(f"Время с lru_cache: {elapsed:.4f}s")
print("Статистика:", expensive_fn.cache_info())

### 3) Подробный разбор кода
- `OrderedDict` хранит порядок вставки; `move_to_end` обновляет «недавность».  
- В `put` элемент либо обновляется, либо добавляется; переполнение вызывает удаление самого старого (`last=False`).
- Декоратор `@lru_cache` автоматически кэширует значения по аргументам и возвращает `cache_info()` (hits/misses).

### 4) Производительность и память
- Операции `get/put` в ручной реализации в среднем O(1).
- Память: O(capacity) + накладные расходы структуры `dict`/`OrderedDict`.
- `lru_cache` может резко ускорить I/O-bound/CPU-bound повторяющиеся вызовы, но увеличивает использование RAM.

### 5) Вопросы с собеседования
1. Чем LRU отличается от LFU и когда LFU лучше?
2. Как сделать LRU cache thread-safe?
3. Почему mutable-аргументы опасны для lru_cache?
4. Как инвалидация кэша влияет на consistency в проде?

### 6) Практические задачи (сложные)
1. Реализуйте LRU без OrderedDict (dict + двусвязный список).
2. Добавьте TTL к каждому ключу.
3. Добавьте метрики hit-rate и auto-tuning maxsize.

### 7) Частые ловушки
- Кэширование функций с побочными эффектами.
- Неочевидный рост памяти при большом maxsize.
- Ключи с высокой кардинальностью приводят к низкому hit-rate.

## Тема 2: Дескрипторы

### 1) Глубокая теория
Дескриптор — объект с методами `__get__`, `__set__`, `__delete__`, который контролирует доступ к атрибуту.

Ключевая идея: атрибут класса может управлять чтением/записью в экземпляр.
- Data descriptor: определяет `__set__` и/или `__delete__`.
- Non-data descriptor: только `__get__`.

Применение: валидация, lazy loading, ORM.

In [ ]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("Значение должно быть положительным")
        setattr(instance, self.private_name, value)

class Product:
    price = PositiveNumber()
    def __init__(self, price):
        self.price = price

p = Product(100)
print(p.price)

### 3) Подробный разбор кода
`__set_name__` связывает имя поля, `__set__` валидирует, `__get__` читает из приватного атрибута экземпляра.

### 4) Производительность и память
Небольшой overhead на доступ, но отличная централизация логики. Один дескриптор разделяется всем классом.

### 5) Вопросы с собеседования
1. Descriptor vs @property?
2. Приоритет data descriptor в lookup?
3. Где это критично в ORM?

### 6) Практические задачи (сложные)
1. Typed descriptor.
2. Descriptor с lazy loading.
3. Логирование чтений/записей.

### 7) Частые ловушки
Shared state в дескрипторе между экземплярами; обход валидации через прямое изменение `_field`.

## Тема 3: Метаклассы

### 1) Глубокая теория
Метакласс — класс, который создаёт классы. Используется для registry, enforce-правил, модификации class namespace.

In [ ]:
class RegistryMeta(type):
    registry = {}
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if name != "BaseModel":
            mcls.registry[name] = cls
        return cls

class BaseModel(metaclass=RegistryMeta):
    pass

class UserModel(BaseModel):
    pass

print(RegistryMeta.registry)

### 3) Подробный разбор кода
Каждый подкласс автоматически регистрируется в словаре метакласса.

### 4) Производительность и память
Цена по CPU мала (создание классов редко), но высокая цена поддержки/читаемости.

### 5) Вопросы с собеседования
1. Когда лучше __init_subclass__?
2. Конфликт метаклассов?
3. Где это оправдано в ML-платформе?

### 6) Практические задачи (сложные)
1. Проверка обязательного метода `predict`.
2. Автогенерация `__repr__`.
3. Реестр с namespace.

### 7) Частые ловушки
Избыточная магия, сложный дебаг.

## Тема 4: Модель памяти Python

### 1) Глубокая теория
CPython: заголовок объекта (refcount, тип) + payload. `del` удаляет имя, не гарантирует немедленное освобождение памяти.

In [ ]:
import sys
x=[1,2,3]; y=x
print(id(x), id(y))
print(sys.getrefcount(x))
del y
print(sys.getrefcount(x))

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 5: Подсчёт ссылок

### 1) Глубокая теория
Refcount — базовый механизм управления памятью в CPython. Ноль ссылок => деаллокация; циклы требуют GC.

In [ ]:
import sys
class A: pass
a=A(); print(sys.getrefcount(a))
b=a; print(sys.getrefcount(a))
del b; print(sys.getrefcount(a))

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 6: Garbage Collection

### 1) Глубокая теория
Generational GC очищает циклические ссылки. Поколения 0/1/2 балансируют частоту сканирования и паузы.

In [ ]:
import gc
class Node:
    def __init__(self): self.ref=None
n1=Node(); n2=Node(); n1.ref=n2; n2.ref=n1
del n1,n2
print('collected=', gc.collect())

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 7: Внутренности объектов

### 1) Глубокая теория
Lookup: data descriptor -> instance __dict__ -> class attrs/non-data descriptor -> bases(MRO).

In [ ]:
class D:
    def __get__(self,inst,owner): return 'descriptor'
    def __set__(self,inst,val): inst.__dict__['x']=val
class C: x=D()
c=C(); c.__dict__['x']='instance'; print(c.x); print(C.__mro__)

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 8: __slots__

### 1) Глубокая теория
`__slots__` уменьшает память на объект за счёт отказа от динамического `__dict__`.

In [ ]:
import sys
class N:
    def __init__(self): self.x=1; self.y=2
class S:
    __slots__=('x','y')
    def __init__(self): self.x=1; self.y=2
n=N(); s=S()
print(sys.getsizeof(n), sys.getsizeof(n.__dict__))
print(sys.getsizeof(s))

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 9: Интернирование

### 1) Глубокая теория
Повторное использование immutable-объектов (особенно строк) может уменьшить память и ускорить сравнения.

In [ ]:
import sys
s1=sys.intern('feature_key')
s2=sys.intern('feature_key')
print(s1 is s2)

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 10: Edge cases в Python

### 1) Глубокая теория
Классика: mutable defaults, late binding, `is` vs `==`, NaN-поведение.

In [ ]:
def append_item(x,bucket=[]):
    bucket.append(x); return bucket
print(append_item(1), append_item(2))
funcs=[lambda i=i: i for i in range(3)]
print([f() for f in funcs])
n1=float('nan'); n2=float('nan'); print(n1==n2, n1 is n2)

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 11: Big-O deep dive

### 1) Глубокая теория
Обсуждайте не только асимптотику, но и амортизацию, константы, кэш-локальность, распределение данных.

In [ ]:
import time
def bench(n=200000):
    arr=list(range(n)); st=set(arr); t=n-1
    a=time.perf_counter(); t in arr; b=time.perf_counter(); t in st; c=time.perf_counter()
    return b-a, c-b
print(bench())

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Тема 12: Частые tricky-вопросы

### 1) Глубокая теория
Глубина интервью — в trade-offs: корректность, latency, memory footprint, поддерживаемость.

In [ ]:
import copy
x=[[1],[2]]; y=copy.copy(x); z=copy.deepcopy(x)
x[0].append(99)
print('x',x,'y',y,'z',z)

### 3) Подробный разбор кода
Ключевая цель примера — продемонстрировать механику темы на минимальном, но показательном кейсе.

### 4) Производительность и память
Оцените временную и пространственную сложность, затем подтвердите профилированием на реалистичных объёмах данных.

### 5) Вопросы с собеседования
1. Объясните внутренний механизм.
2. Какие trade-offs в production?
3. Какие риски/edge cases?

### 6) Практические задачи (сложные)
1. Усложните пример до production-подобного сценария.
2. Добавьте тесты на крайние случаи.
3. Снимите метрики времени и памяти.

### 7) Частые ловушки
Нельзя опираться на поведение CPython как на универсальную гарантию для всех реализаций Python.

## Mock Interview

Ниже 20 сложных смешанных вопросов уровня продвинутого собеседования.

1. Спроектируйте кэш для feature engineering: почему LRU, а не LFU? Какие метрики выберете для auto-tuning?
2. Как реализовать LRU с TTL и избежать race conditions в многопоточном API?
3. Объясните порядок разрешения атрибутов в Python и роль data descriptor в этом процессе.
4. Когда вы выберете descriptor вместо property в промышленном коде?
5. Метакласс vs __init_subclass__: где граница применимости и как снизить сложность решения?
6. Как обнаружить и объяснить конфликт метаклассов при множественном наследовании?
7. Опишите модель памяти CPython: где хранится refcount и почему это важно для производительности?
8. Почему `del obj` не всегда освобождает память немедленно? Приведите несколько сценариев.
9. Как циклические ссылки взаимодействуют с reference counting и поколенческим GC?
10. Какие риски несёт `__del__` в объектах, участвующих в циклах?
11. Как бы вы отлаживали утечку памяти в долгоживущем ML-сервисе?
12. В каких случаях `__slots__` даст существенную выгоду, а в каких — почти нет?
13. Объясните ограничения `__slots__` при наследовании и интеграции со сторонними библиотеками.
14. Что такое интернирование строк и когда `sys.intern` оправдан в реальной системе?
15. Почему использование `is` для строк/чисел считается ловушкой? Когда `is` действительно корректен?
16. Разберите подводные камни mutable default arguments и предложите безопасный шаблон API.
17. Почему `nan != nan` и как это влияет на качество проверки данных в ML-пайплайнах?
18. Сравните Big-O и реальную производительность: когда O(n) может победить O(log n)?
19. Какие операции dict/set имеют худший случай O(n), и как это учитывать в security/reliability?
20. Дайте пример, когда оптимизация времени ухудшает память (и наоборот), и как принять инженерное решение.